In [0]:
# Instead of merging constantly, we are going to just over write everything since our data set isn't that large

import pyspark.sql.functions as F
from pyspark.sql.types import *


In [0]:
%run "../utilities/Delta Merge"

In [0]:
silver_data_source = spark.read.table('lakehouse.`03_silver`.earthquake_events')

In [0]:
silver_data_source.display()

In [0]:
# TODO
# 1. event_density per location(lat and long)

In [0]:
# Clone from seismic activities notebook, just changing the values we want
# Grouping by lat and log and rounding the number becuase they are to long(round to 1 decimal point)
event_density = (
    silver_data_source
    .groupBy(
        F.round(F.col('latitude'), 1).alias('avg_latitude'),
        F.round(F.col('longitude'), 1).alias('avg_longitude'),
    ) 
    .agg(
        F.count("*").alias("event_density"),
        F.avg("mag").alias("avg_mag"),
        F.max("mag").alias("max_mag")
    )
     .withColumn('hash_id', F.sha2(F.concat_ws('_', F.col('avg_latitude'), F.col('avg_longitude')),256).substr(0,15)) # -> using substring to shorten the hash
)
event_density.display()


In [0]:
table_name = 'lakehouse.`04_gold`.event_density'

In [0]:
if check_delta_table(table_name):
    print(f"{table_name} table exists in catalog, updating it")
    delta_merge(event_density, table_name, "hash_id")
else:
    print(f"{table_name} table does not exist in catalog, writing it for first time")
    event_density.write.mode("overwrite").format("delta").option("delta.enableChangeDataFeed", "true").saveAsTable(table_name)